# 🌷 LILY VIDEO STUDIO — clean Kaggle build

Fresh-session version. Turn on a Kaggle GPU, then run the three cells below from top to bottom. The last cell launches Lily's simple iPhone UI.

Models in the UI: **LTX-2.3 Distilled 1.1**, **Wan 2.2 I2V**, and **HunyuanVideo 1.5**. Each checkpoint downloads automatically the first time you use that model in a fresh Kaggle session.


## 1 — Build the playground


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil
import torch

subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.is_available(), 'NO GPU — in Kaggle turn on Settings → Accelerator → GPU.'
print('✅ GPU:', torch.cuda.get_device_name(0))
print('✅ Torch:', torch.__version__, '| CUDA:', torch.version.cuda)

ROOT = Path('/kaggle/working/Wan2GP')
DATA = Path('/kaggle/temp/Wan2GP-data')
CKPTS = DATA / 'ckpts'
LORAS = DATA / 'loras'
CACHE = DATA / 'cache'
OUTPUTS = Path('/kaggle/working/Wan2GP-outputs')
for p in (DATA, CKPTS, LORAS, CACHE, OUTPUTS): p.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(CACHE / 'huggingface')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(CACHE / 'huggingface' / 'hub')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE / 'huggingface' / 'transformers')
os.environ['TORCH_HOME'] = str(CACHE / 'torch')
os.environ['XDG_CACHE_HOME'] = str(CACHE / '.cache')
os.environ['WAN_CACHE_DIR'] = str(CACHE)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

REPO = 'https://github.com/deepbeepmeep/Wan2GP.git'
if not (ROOT / '.git').exists():
    if ROOT.exists(): shutil.rmtree(ROOT)
    subprocess.run(['git','clone','--depth','1',REPO,str(ROOT)], check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'], check=False)

def attach(repo_dir, storage_dir):
    storage_dir.mkdir(parents=True, exist_ok=True)
    if repo_dir.is_symlink():
        if repo_dir.resolve() == storage_dir.resolve(): return
        repo_dir.unlink()
    elif repo_dir.exists():
        for item in list(repo_dir.iterdir()):
            dest = storage_dir / item.name
            if not dest.exists(): shutil.move(str(item), str(dest))
        shutil.rmtree(repo_dir)
    repo_dir.symlink_to(storage_dir, target_is_directory=True)

attach(ROOT/'ckpts', CKPTS)
attach(ROOT/'loras', LORAS)
attach(ROOT/'outputs', OUTPUTS)

env = os.environ.copy(); env['DEBIAN_FRONTEND'] = 'noninteractive'
prefix = [] if os.geteuid() == 0 else ['sudo']
subprocess.run(prefix + ['apt-get','update','-qq'], check=True, env=env)
subprocess.run(prefix + ['apt-get','install','-y','--no-install-recommends','ffmpeg','libglib2.0-0','libgl1','libportaudio2'], check=True, env=env)
print('✅ WanGP cloned and system libraries ready.')


## 2 — Install WanGP

This is the long one. Let it finish completely. It preserves Kaggle's CUDA-enabled Torch instead of replacing it.


In [ ]:
import importlib, subprocess, sys, os
import torch

installed = {'torch': torch.__version__}
for name in ('torchvision','torchaudio'):
    try: installed[name] = importlib.import_module(name).__version__
    except Exception: installed[name] = None

constraints = CACHE / 'kaggle-torch-constraints.txt'
pins = [f"torch=={installed['torch'].split('+',1)[0]}"]
for name in ('torchvision','torchaudio'):
    if installed[name]: pins.append(f"{name}=={installed[name].split('+',1)[0]}")
constraints.write_text('\n'.join(pins)+'\n')
print('Protecting Kaggle Torch with constraints:')
print(constraints.read_text())

env = os.environ.copy(); env['PIP_NO_CACHE_DIR'] = '1'
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--upgrade','setuptools','wheel'], check=True, env=env)
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','--upgrade-strategy','only-if-needed','-r',str(ROOT/'requirements.txt'),'-c',str(constraints)], check=True, env=env)

target = ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
if target.exists():
    text = target.read_text(); patched = text.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')")
    if patched != text: target.write_text(patched)

for pkg in ('mmgp','rembg','gradio'):
    importlib.import_module(pkg)
assert torch.cuda.is_available()
print('✅ EVERYTHING INSTALLED. GPU still alive:', torch.cuda.get_device_name(0))


## 3 — 🌷 Launch Lily Video Studio

Keep this cell running. When a public **gradio.live** link appears, tap it on your iPhone.


In [ ]:
import urllib.request, subprocess, sys, os

STUDIO = ROOT / 'lily_video_studio.py'
URL = 'https://raw.githubusercontent.com/benruiz1024-ops/hi/main/lily_video_studio.py'
urllib.request.urlretrieve(URL, STUDIO)
print('🌷 Lily Video Studio updated from GitHub.')
print('Launching… wait for the public gradio.live link.')
os.chdir(ROOT)
subprocess.run([sys.executable,'-u',str(STUDIO)], check=False)
